In [1]:
import pandas as pd

# Load raw dataset for initial inspection only — no cleaning at this stage
df = pd.read_csv("../data/raw/a_steam_data_2021_2025.csv")

print("File loaded successfully.")

File loaded successfully.


In [2]:
# 1. Shape and column overview
print("Shape:", df.shape)
print("\nColumn dtypes:\n", df.dtypes)

Shape: (65521, 10)

Column dtypes:
 appid                int64
name                object
release_year         int64
release_date        object
genres              object
categories          object
price              float64
recommendations      int64
developer           object
publisher           object
dtype: object


In [3]:
# 2. First rows to visually inspect structure
df.head(10)

,appid,name,release_year,release_date,genres,categories,price,recommendations,developer,publisher
0,3057270,Seafarer's Gambit,2024,"Jul 5, 2024",Action;Adventure;Indie;RPG;Strategy,Single-player;Family Sharing,3.99,0,Bouncy Rocket Studios,Bouncy Rocket Studios
1,3822840,Capitalist Misadventures,2025,"Jul 25, 2025",Casual;Indie;Simulation;Strategy,Single-player;Save Anytime;Family Sharing,7.99,0,Caramelo Studios,Caramelo Studios
2,3216640,The Beast and the Princess,2025,"Jun 17, 2025",Adventure;Indie;Strategy,Single-player;Steam Achievements;Full controll...,12.99,0,Libragames,Libragames
3,2403620,Air Twister,2023,"Nov 10, 2023",Action;Adventure;Indie,Single-player;Steam Achievements;Full controll...,24.99,0,YS Net,ININ
4,1538040,Horde Slayer,2021,"Mar 19, 2021",Action;Adventure;Casual;Indie;RPG;Early Access,Single-player;Steam Achievements;Full controll...,3.99,0,Wagner Rodrigues,Wagner Rodrigues
5,1724980,The Lone Blade,2023,"May 23, 2023",Action;Adventure;Indie,Single-player;Full controller support;Family S...,1.99,0,Opia Games,Opia Games;Plug In Digital
6,3822820,Knight Crawler,2025,"Jul 16, 2025",Action;Indie;Free To Play,Single-player;Full controller support;Custom V...,0.00,0,Taylor Conolley,Taylor Conolley
7,3863460,No Sweet Looks,2025,"Aug 28, 2025",Action;Indie,Single-player;Steam Achievements;Full controll...,2.99,0,halvardo13,halvardo13
8,3216610,League Of Tacticians: Path of Tarkan,2025,"May 21, 2025",Adventure;RPG;Strategy,Single-player;Family Sharing,3.99,0,Oba Games,Oba Games
9,3057250,Pennylooter,2025,"Sep 8, 2025",Action;Indie,Single-player;Steam Achievements;Full controll...,6.99,0,Josh Sellers,Josh Sellers


In [4]:
# 3. Missing values per column
print("Missing values per column:\n", df.isnull().sum())

Missing values per column:
 appid                0
name                 0
release_year         0
release_date         0
genres              66
categories           7
price                0
recommendations      0
developer           53
publisher          183
dtype: int64


In [5]:
# 4. Full duplicate rows check
print("Full duplicate rows:", df.duplicated().sum())

Full duplicate rows: 0


In [6]:
# 5. Column names, listed explicitly for reference
print(list(df.columns))

['appid', 'name', 'release_year', 'release_date', 'genres', 'categories', 'price', 'recommendations', 'developer', 'publisher']


In [7]:
# Check whether appid is truly unique — this is the candidate primary key
print("Unique appid count:", df["appid"].nunique())
print("Total rows:", len(df))
print("Duplicate appid rows:", df["appid"].duplicated().sum())

Unique appid count: 65521
Total rows: 65521
Duplicate appid rows: 0


In [8]:
# Distribution of recommendations — checking if zeros dominate suspiciously
print(df["recommendations"].describe())
print("\nRows with recommendations == 0:", (df["recommendations"] == 0).sum())
print("Percentage of zeros:", round((df["recommendations"] == 0).mean() * 100, 2), "%")

count     65521.000000
mean        362.165336
std        6936.837198
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max      862487.000000
Name: recommendations, dtype: float64

Rows with recommendations == 0: 57515
Percentage of zeros: 87.78 %


In [9]:
# Cross-check: does price == 0 always align with the "Free To Play" tag?
free_by_price = df["price"] == 0
free_by_tag = df["genres"].str.contains("Free To Play", na=False)

print("Games with price == 0:", free_by_price.sum())
print("Games tagged 'Free To Play':", free_by_tag.sum())
print("Games where both agree:", (free_by_price & free_by_tag).sum())
print("Price == 0 but NOT tagged Free To Play:", (free_by_price & ~free_by_tag).sum())
print("Tagged Free To Play but price > 0:", (~free_by_price & free_by_tag).sum())

Games with price == 0: 11962
Games tagged 'Free To Play': 8043
Games where both agree: 7953
Price == 0 but NOT tagged Free To Play: 4009
Tagged Free To Play but price > 0: 90


In [10]:
# Convert release_date to check the real temporal range of the dataset
df["release_date_parsed"] = pd.to_datetime(df["release_date"], errors="coerce")

print("Min date:", df["release_date_parsed"].min())
print("Max date:", df["release_date_parsed"].max())
print("\nRows with future release date (after today):")
print((df["release_date_parsed"] > pd.Timestamp.now()).sum())

print("\nRows where date failed to parse:", df["release_date_parsed"].isna().sum())

Min date: 2021-01-01 00:00:00
Max date: 2025-12-31 00:00:00

Rows with future release date (after today):
0

Rows where date failed to parse: 1400


In [11]:
# Inspect raw values of release_date that failed to parse
failed_dates = df[df["release_date_parsed"].isna()]
print("Sample of raw release_date values that failed to parse:")
print(failed_dates["release_date"].value_counts().head(20))

# Check whether release_year is still populated for these rows
print("\nrelease_year values for failed rows:")
print(failed_dates["release_year"].value_counts())

Sample of raw release_date values that failed to parse:
release_date
2025             836
Q4 2025          355
December 2025    209
Name: count, dtype: int64

release_year values for failed rows:
release_year
2025    1400
Name: count, dtype: int64


In [12]:
# Inspect the 90 rows tagged Free To Play but with price > 0
anomaly = df[(df["genres"].str.contains("Free To Play", na=False)) & (df["price"] > 0)]
anomaly[["appid", "name", "price", "genres"]].head(15)

,appid,name,price,genres
312,3215270,Place of Decay,4.99,Action;Adventure;Indie;Strategy;Free To Play
562,1340180,Sherwood Extreme,9.99,Action;Adventure;Casual;Indie;RPG;Free To Play
788,3213850,gogh: Focus with Your Avatar,11.99,Action;Adventure;Casual;Indie;RPG;Simulation;S...
1047,1534050,SCP: Breakout,11.99,Action;Adventure;Free To Play;Massively Multip...
3426,3007590,Dog Walking Adventures,2.99,Adventure;Casual;Indie;Simulation;Free To Play
5038,2390670,ASSASSIN: The First List (Beta),6.99,Action;Indie;Strategy;Free To Play
5105,3003300,ZM Desktop Elf,3.99,Casual;Free To Play;Utilities
5916,3849380,Urban Hunter,29.99,Action;Adventure;Free To Play
7604,3367740,the secret story : classic edition,1.99,Action;Adventure;Indie;RPG;Simulation;Free To ...
7717,2382310,The Last Society,0.99,Action;Adventure;RPG;Strategy;Free To Play
